# TP Jour 2 — Modèle d'Exécution Spark, Jointures Multi-Datasets & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze** (issues d'un ou **plusieurs datasets Kaggle**), nettoyer les anomalies, typer fortement les colonnes et harmoniser les **clés de jointure inter-sources**.
3. Mettre en œuvre une **jointure distribuée multi-datasets maîtrisée** (`BroadcastHashJoin` via `F.broadcast` pour un référentiel/dataset compact, ou `SortMergeJoin` entre deux grandes sources) **en contrôlant la cardinalité (absence d'explosion N-à-M)**.
4. Sauvegarder votre table analytique enrichie dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).
5. Mener une **Analyse Exploratoire des Données (EDA)** et produire des **visualisations 1D, 2D et multidimensionnelles (ACP / PCA)** sur la couche Silver.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

## 1. Chargement de vos tables Bronze (Multi-Datasets / Multi-Tables) & Diagnostic Qualité

### Exercice 1.1 — Charger toutes vos tables Bronze et profiler valeurs manquantes / doublons
1. Chargez dans `bronze_dfs` votre table de faits principale ainsi que la (ou les) table(s) secondaire(s) à joindre depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` sur vos colonnes clés (en particulier les **clés de jointure**) ainsi que les doublons éventuels.

In [ ]:
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = spark._jvm.org.apache.hadoop.fs.Path

bronze_dfs = {}
for s in sorted(fs.listStatus(PathHDFS("/lakehouse/bronze")), key=lambda x: x.getPath().getName()):
    name = s.getPath().getName()
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log")):
        df = spark.read.format("delta").load(f"/lakehouse/bronze/{name}")
        bronze_dfs[name] = df
        print(f"[{name:<42}] -> {df.count():>9,} lignes | {df.columns[:5]}")

for name, df in bronze_dfs.items():
    cols = [c for c in df.columns if not c.startswith("_")]
    print(f"\n=== Valeurs nulles : {name} ===")
    df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in cols]).show(truncate=False)

## 2. Harmonisation Inter-Datasets, Nettoyage & Jointure Distribuée (`BroadcastHashJoin` / `SortMergeJoin`)

### Bonnes pratiques de jointure entre plusieurs datasets hétérogènes :
1. **Harmonisation des clés de jointure (*Key Alignment*)** : deux datasets Kaggle d'auteurs différents n'ont presque jamais exactement le même format de clé :
   - *Axe temporel* : ramenez les horodatages à la même granularité et au même fuseau (`F.to_date(...)`, `F.date_trunc("hour", ...)` ou format `yyyy-MM`).
   - *Axe géographique / catégoriel* : normalisez la casse, les espaces et les codes (`F.upper(F.trim(col))`, code ISO pays/région, code postal ou grille `F.round(lat, 2)`).
2. **Prévention des explosions N-à-M** : avant un `LEFT JOIN` de la table principale (`N` lignes) vers un second dataset, assurez-vous que la clé de jointure est **unique côté table secondaire** via une pré-agrégation métier (`groupBy(join_keys).agg(...)`) ou `dropDuplicates(join_keys)` ! Vérifiez systématiquement que le nombre de lignes après `LEFT JOIN` n'explose pas.
3. **Choix de la stratégie physique Spark** :
   - **Table secondaire compacte (< quelques dizaines de Mo)** : forcez **`F.broadcast(df_sec)`** pour obtenir un `BroadcastHashJoin` sans *Shuffle* de la grande table.
   - **Deux grands datasets volumineux** : laissez Spark exécuter un `SortMergeJoin` distribué (optimisé dynamiquement par *Adaptive Query Execution*).

### Exercice 2.1 — Pipeline complet de Data Wrangling vers `/lakehouse/silver/`

Conformément à la méthodologie du cours **[Chapitre 2 — Data Wrangling](https://datascience.apti.space/cours/2_wrangling/)**, nous déroulons le nettoyage et l'enrichissement en **3 étapes rigoureuses** :
1. **Diagnostic de la Typologie du Vide (§2.1)** : distinguer les mécanismes **MCAR**, **MAR** et **MNAR**, et débusquer les **vides masqués** (chaînes vides `""` / `"null"` et **chiffres sentinelles `0`**).
2. **Traitement, Harmonisation & Validation de Domaine (§2.1.3 & §2.3)** : suppression des colonnes vides à > 90 %, **flags binaires d'absence** avant imputation (médiane pour le quantitatif asymétrique, modalité explicite pour le qualitatif), nettoyage typographique (`lower` + `trim`), standardisation temporelle ISO 8601 (contrôle du piège Epoch `1970-01-01` + feature `is_weekend`) et contrôle du **domaine de définition**.
3. **Transformations Robustes, Encodage & Jointures Multi-Datasets (§2.4 & §2.5)** : pré-agrégations anti-explosion N-à-M, encodage *One-Hot* basse cardinalité (`payment_type`), confrontation **IQR vs Z-Score** + *Robust Scaling* sur les montants/délais, `BroadcastHashJoin` inter-datasets et sauvegarde Delta dans `/lakehouse/silver/`.

#### Étape 2.1.1 — Analyse des Valeurs Manquantes (MCAR, MAR, MNAR) & Détection des Vides Masqués

> *« Avant de boucher un trou, l'enquêteur doit se demander : pourquoi cet indice a-t-il été effacé ? L'absence de donnée est en soi une donnée. »* (§2.1.1)

Vérifions empiriquement sur nos tables Bronze les **4 cas d'école** du cours :
- **MAR (*Missing At Random*)** sur `orders.order_delivered_customer_date` : l'absence s'explique-t-elle par une autre variable observée (`order_status` : commandes en cours ou annulées) ?
- **MNAR (*Missing Not At Random*)** sur `order_reviews.review_comment_message` : la probabilité de laisser un commentaire dépend-elle directement du niveau d'insatisfaction (`review_score`) ?
- **MCAR (*Missing Completely At Random*)** sur `products` : seulement 2 poids/dimensions manquants (< 0,01 %) et 610 catégories manquantes (1,85 %).
- **Vide masqué par chiffre sentinelle (`0`)** sur `closed_deals.declared_monthly_revenue` : `0` NULL détecté techniquement, mais combien de marchands ont en réalité `0.0` (revenu non déclaré) ?

In [ ]:
orders_bz = bronze_dfs["olist_orders_dataset"]
items_bz = bronze_dfs["olist_order_items_dataset"]
payments_bz = bronze_dfs["olist_order_payments_dataset"]
reviews_bz = bronze_dfs["olist_order_reviews_dataset"]
customers_bz = bronze_dfs["olist_customers_dataset"]
products_bz = bronze_dfs["olist_products_dataset"]
cat_trans_bz = bronze_dfs["product_category_name_translation"]
leads_bz = bronze_dfs["olist_marketing_qualified_leads_dataset"]
closed_deals_bz = bronze_dfs["olist_closed_deals_dataset"]

SENTINEL_STRINGS = ["", "null", "none", "n/a", "na", "nan", "?", "unknown"]

# 1. Synthèse globale des taux de manque réels (NULL + chaînes vides/trompeuses)
print("=== 1. SYNTHÈSE DES COLONNES PRÉSENTANT DES MANQUES (RÉELS OU CHAÎNES MASQUÉES) ===")
missing_rows = []
for tbl_name, df in bronze_dfs.items():
    total_rows = df.count()
    exprs = []
    data_cols = [c for c, _ in df.dtypes if not c.startswith("_")]
    for col_name, dtype in df.dtypes:
        if col_name.startswith("_"):
            continue
        cond = F.col(col_name).isNull()
        if dtype == "string":
            cond = cond | F.lower(F.trim(F.col(col_name))).isin(SENTINEL_STRINGS)
        exprs.append(F.count(F.when(cond, 1)).alias(col_name))
    counts_row = df.select(exprs).first().asDict()
    for col_name, n_miss in counts_row.items():
        if n_miss > 0:
            pct = round(100.0 * n_miss / total_rows, 2)
            missing_rows.append((tbl_name, col_name, n_miss, total_rows, pct))

spark.createDataFrame(
    sorted(missing_rows, key=lambda r: r[4], reverse=True),
    ["table", "colonne", "nb_manquants", "total_lignes", "pct_manquant"],
).show(25, truncate=False)

# 2. Preuve d'un mécanisme MAR : `order_delivered_customer_date` manquant selon `order_status`
print("=== 2. DIAGNOSTIC MAR : Absence de `order_delivered_customer_date` expliquée par `order_status` ===")
(
    orders_bz.groupBy("order_status")
    .agg(
        F.count("*").alias("nb_commandes"),
        F.count(F.when(F.col("order_delivered_customer_date").isNull(), 1)).alias("date_livraison_manquante"),
        F.round(100.0 * F.avg(F.when(F.col("order_delivered_customer_date").isNull(), 1.0).otherwise(0.0)), 2).alias("pct_manquant"),
    )
    .orderBy(F.desc("date_livraison_manquante"))
    .show(truncate=False)
)

# 3. Preuve d'un mécanisme MNAR : absence de `review_comment_message` corrélée à la satisfaction (`review_score`)
print("=== 3. DIAGNOSTIC MNAR : Taux d'absence du commentaire client selon la note `review_score` ===")
(
    reviews_bz.filter(F.col("review_score").between(1, 5))
    .groupBy("review_score")
    .agg(
        F.count("*").alias("nb_avis"),
        F.round(
            100.0 * F.avg(
                F.when(F.col("review_comment_message").isNull() | (F.trim(F.col("review_comment_message")) == ""), 1.0).otherwise(0.0)
            ),
            2,
        ).alias("pct_commentaire_absent"),
    )
    .orderBy("review_score")
    .show(truncate=False)
)

# 4. Preuve d'un Vide Masqué par Chiffre Sentinelle (`0.0`) dans `closed_deals.declared_monthly_revenue`
print("=== 4. DÉTECTION DE CHIFFRE SENTINELLE (`0.0`) : `closed_deals.declared_monthly_revenue` ===")
(
    closed_deals_bz.select(
        F.count("*").alias("total_deals"),
        F.count(F.when(F.col("declared_monthly_revenue").isNull(), 1)).alias("nb_null_officiels"),
        F.count(F.when(F.col("declared_monthly_revenue") == 0, 1)).alias("nb_zeros_sentinelles"),
        F.round(100.0 * F.avg(F.when(F.col("declared_monthly_revenue") == 0, 1.0).otherwise(0.0)), 2).alias("pct_zeros_masques"),
        F.round(F.avg("declared_monthly_revenue"), 2).alias("moyenne_biaisee_avec_0"),
        F.round(F.avg(F.when(F.col("declared_monthly_revenue") > 0, F.col("declared_monthly_revenue"))), 2).alias("moyenne_reelle_hors_0"),
    )
    .show(truncate=False)
)

#### Bilan du diagnostic & décisions de traitement (selon §2.1.3, §2.3 & §2.4 du cours)

| Table / Colonne | Taux d'absence | Mécanisme identifié | Stratégie appliquée (Préconisations du cours) |
|---|---|---|---|
| `closed_deals` (`has_company`, `has_gtin`, `average_stock`, `declared_product_catalog_size`) | **91 % – 93 %** | Champs optionnels quasi vides | **Suppression de colonnes** (> 90 % vides, inexploitables). |
| `closed_deals.declared_monthly_revenue` | **0 % NULL mais 94,7 % de `0.0`** | **Vide masqué (Chiffre sentinelle `0`) + MNAR** | Conversion `0 -> NULL` + **Flag binaire** `has_declared_revenue` (moyenne biaisée 18,7× sinon : 73k vs 1,37M BRL !). |
| `order_reviews.review_comment_message` | **58,7 %** (64–69 % à 4–5★ vs 23,5 % à 1★) | **MNAR** (les mécontents écrivent ~3× plus souvent) | **Flag binaire d'absence/présence** `has_review_comment` (`0/1`) préservant le signal comportemental. |
| `orders.order_delivered_customer_date` | **2,98 %** (99–100 % hors statut `delivered`, 0,01 % sur `delivered`) | **MAR** (expliqué par `order_status`) | Conservation du `NaT`/`NULL` temporel + **Flag binaire** `is_delivered` (aucun `dropna` destructif). |
| `products` (`product_weight_g`, dimensions) | **0,006 %** (2 lignes / 32 951) | **MCAR** (oubli isolé de saisie catalogue) | **Imputation par la Médiane** (`700 g`, `25×13×20 cm`, robuste aux produits lourds). |
| `products.product_category_name` & `leads.origin` | **1,85 %** et chaînes `"unknown"` | **MCAR / Vide masqué textuel** | Détection de `"unknown"` + **Flags binaires** (`category_is_missing`, `origin_is_missing`) + modalité `"non_communique"`. |

---
#### Étape 2.1.2 — Nettoyage Typographique, Imputation, Standardisation Temporelle ISO 8601 & Validation de Domaine

In [ ]:
def clean_str(col_name: str):
    """Harmonisation typographique (§2.3.1) : trim + minuscules + conversion des chaînes vides/sentinelles en vrai NULL."""
    norm = F.regexp_replace(F.lower(F.trim(F.col(col_name))), r"\s+", " ")
    return F.when(norm.isNull() | norm.isin(SENTINEL_STRINGS), F.lit(None)).otherwise(norm)

# 1. PRODUITS : Flag binaire `category_is_missing` + Imputation par la Médiane sur poids/dimensions (MCAR < 0,01%) + Broadcast Join traduction
num_prod_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
prod_medians = products_bz.select([
    F.percentile_approx(F.when(F.col(c) > 0, F.col(c)), 0.5).alias(c) for c in num_prod_cols
]).first().asDict()
print("Médianes calculées pour l'imputation robuste `products` :", prod_medians)

cat_trans_clean = (
    cat_trans_bz
    .withColumn("product_category_name", clean_str("product_category_name"))
    .withColumn("product_category_name_english", clean_str("product_category_name_english"))
    .dropna(subset=["product_category_name"])
    .dropDuplicates(["product_category_name"])
)

products_clean = (
    products_bz
    .drop("_ingested_at", "_source_dataset", "_source_file")
    .dropDuplicates(["product_id"])
    .withColumn("product_category_name", clean_str("product_category_name"))
    .withColumn("category_is_missing", F.when(F.col("product_category_name").isNull(), F.lit(1)).otherwise(F.lit(0)))
    .join(
        F.broadcast(cat_trans_clean.select("product_category_name", "product_category_name_english")),
        on="product_category_name",
        how="left",
    )
    .withColumn(
        "category",
        F.coalesce(F.col("product_category_name_english"), F.col("product_category_name"), F.lit("non_communique")),
    )
)
for c, med_val in prod_medians.items():
    products_clean = products_clean.withColumn(
        c, F.when((F.col(c).isNull()) | (F.col(c) <= 0), F.lit(float(med_val))).otherwise(F.col(c).cast("double"))
    )

print("\n=== Vérification `products_clean` (aperçu des produits dont la catégorie était manquante) ===")
products_clean.filter(F.col("category_is_missing") == 1).select(
    "product_id", "category", "category_is_missing", "product_weight_g", "product_length_cm"
).show(5, truncate=False)

# 2. DATASET #2 (B2B Marketing Funnel) : Suppression des colonnes >90% vides, correction du 0 sentinelle & dédoublonnage `seller_id`
leads_clean = (
    leads_bz
    .withColumn("origin_clean", clean_str("origin"))
    .withColumn("origin_is_missing", F.when(F.col("origin_clean").isNull(), F.lit(1)).otherwise(F.lit(0)))
    .withColumn("seller_lead_origin", F.coalesce(F.col("origin_clean"), F.lit("non_communique")))
    .select("mql_id", "seller_lead_origin", "origin_is_missing")
    .dropDuplicates(["mql_id"])
)

seller_acquisition = (
    closed_deals_bz
    # Suppression explicite des colonnes vides à > 90 % (`has_company`, `has_gtin`, `average_stock`, `declared_product_catalog_size`)
    .filter(F.col("seller_id").isNotNull())
    .withColumn(
        "declared_monthly_revenue_clean",
        F.when(F.col("declared_monthly_revenue") > 0, F.col("declared_monthly_revenue").cast("double")).otherwise(F.lit(None)),
    )
    .withColumn("has_declared_revenue", F.when(F.col("declared_monthly_revenue_clean").isNotNull(), F.lit(1)).otherwise(F.lit(0)))
    .withColumn("seller_business_segment", F.coalesce(clean_str("business_segment"), F.lit("non_communique")))
    .join(F.broadcast(leads_clean), on="mql_id", how="left")
    .withColumn("seller_lead_origin", F.coalesce(F.col("seller_lead_origin"), F.lit("non_communique")))
    .select("seller_id", "seller_lead_origin", "seller_business_segment", "has_declared_revenue", "declared_monthly_revenue_clean")
    .dropDuplicates(["seller_id"])
)

print("=== Aperçu du référentiel B2B nettoyé `seller_acquisition` (0 sentinelle converti en NULL + flag binaire) ===")
seller_acquisition.orderBy(F.desc("has_declared_revenue")).show(6, truncate=False)

# 3. CLIENTS : Harmonisation typographique villes (minuscules/trim) et codes État ISO (MAJUSCULES)
customers_clean = (
    customers_bz
    .dropDuplicates(["customer_id"])
    .withColumn("customer_city", F.coalesce(clean_str("customer_city"), F.lit("non_communique")))
    .withColumn("customer_state", F.upper(F.trim(F.col("customer_state"))))
    .select("customer_id", "customer_unique_id", "customer_city", "customer_state")
)

# 4. VALIDATION DE DOMAINE (§2.3.3) : Audit chiffré des violations physiques et temporelles avant filtrage
orders_typed = (
    orders_bz
    .dropDuplicates(["order_id"])
    .withColumn("order_status", clean_str("order_status"))
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
    .withColumn(
        "delivery_days",
        F.round((F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_purchase_timestamp")) / 86400.0, 2),
    )
)

domain_audit = [
    ("order_items : prix <= 0 ou fret < 0", items_bz.filter((F.col("price") <= 0) | (F.col("freight_value") < 0)).count()),
    ("order_payments : montant < 0", payments_bz.filter(F.col("payment_value") < 0).count()),
    ("order_reviews : note hors [1, 5]", reviews_bz.filter(~F.col("review_score").between(1, 5)).count()),
    ("orders : livraison antérieure à l'achat (delivery_days < 0)", orders_typed.filter(F.col("delivery_days") < 0).count()),
    ("orders : date fantôme Epoch 1970 (year <= 1970)", orders_typed.filter(F.year("order_purchase_timestamp") <= 1970).count()),
]
print("=== Audit de Validation de Domaine (§2.3.3) ===")
spark.createDataFrame(domain_audit, ["regle_domaine", "nb_violations_detectees"]).show(truncate=False)

#### Étape 2.1.3 — Jointures Multi-Datasets (`BroadcastHashJoin`), Encodage, Outliers (IQR vs Z-Score) & Écriture Silver

Nous assemblons maintenant `/lakehouse/silver/order_items` et la table maîtresse `SILVER_MAIN_PATH = "/lakehouse/silver/orders"` en appliquant :
- **Pré-agrégation à la maille `order_id`** sur `order_items`, `order_payments` (avec **One-Hot Encoding** basse cardinalité sur `payment_type`, §2.5.1) et `order_reviews` (avec le flag MNAR `has_review_comment`).
- **Jointure inter-datasets optimisée** (`F.broadcast(seller_acquisition)` sur `seller_id`) et contrôle du plan physique `.explain("simple")`.
- **Confrontation IQR vs Z-Score (§2.4.2)** sur `delivery_days` et `total_amount` : création du flag `is_delivery_outlier_iqr` ($> Q_3 + 1.5 \times \text{IQR}$) et mise à l'échelle robuste (`total_amount_robust_scaled` = $(X - \text{Médiane}) / \text{IQR}$).

In [ ]:
SILVER_MAIN_PATH = "/lakehouse/silver/orders"

# 1. Table Silver détaillée `/lakehouse/silver/order_items` (BroadcastHashJoin produits + acquisition B2B)
silver_items = (
    items_bz
    .filter((F.col("price") > 0) & (F.col("freight_value") >= 0))
    .dropDuplicates(["order_id", "order_item_id"])
    .join(
        F.broadcast(products_clean.select("product_id", "category", "category_is_missing", "product_weight_g")),
        on="product_id",
        how="left",
    )
    .join(
        F.broadcast(seller_acquisition.select("seller_id", "seller_lead_origin", "seller_business_segment")),
        on="seller_id",
        how="left",
    )
    .withColumn("category", F.coalesce(F.col("category"), F.lit("non_communique")))
    .withColumn("category_is_missing", F.coalesce(F.col("category_is_missing"), F.lit(1)))
    .withColumn("has_b2b_profile", F.when(F.col("seller_lead_origin").isNotNull(), F.lit(1)).otherwise(F.lit(0)))
    .withColumn("seller_lead_origin", F.coalesce(F.col("seller_lead_origin"), F.lit("organic_or_legacy")))
    .withColumn("seller_business_segment", F.coalesce(F.col("seller_business_segment"), F.lit("unspecified")))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double"))
    .withColumn("line_total", F.round(F.col("price") + F.col("freight_value"), 2))
    .select(
        "order_id", "order_item_id", "product_id", "seller_id",
        "category", "category_is_missing", "has_b2b_profile", "seller_lead_origin", "seller_business_segment",
        "product_weight_g", "price", "freight_value", "line_total"
    )
)

print("=== Plan physique de `silver_items` (vérification BroadcastHashJoin) ===")
silver_items.explain("simple")

silver_items.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/lakehouse/silver/order_items")

# 2. Pré-agrégations à la maille `order_id` (zéro explosion N-M) + One-Hot Encoding sur `payment_type` (§2.5.1)
items_per_order = (
    silver_items.groupBy("order_id").agg(
        F.count("*").alias("total_items"),
        F.round(F.sum("price"), 2).alias("items_amount"),
        F.round(F.sum("freight_value"), 2).alias("freight_amount"),
        F.round(F.sum("line_total"), 2).alias("total_amount"),
        F.first("category", ignorenulls=True).alias("category"),
        F.max("category_is_missing").alias("category_is_missing"),
        F.max("has_b2b_profile").alias("has_b2b_profile"),
        F.first("seller_lead_origin", ignorenulls=True).alias("seller_lead_origin"),
        F.first("seller_business_segment", ignorenulls=True).alias("seller_business_segment"),
    )
)

payments_per_order = (
    payments_bz
    .filter(F.col("payment_value") >= 0)
    .withColumn("payment_type", clean_str("payment_type"))
    .groupBy("order_id")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("paid_amount"),
        F.max("payment_installments").alias("max_installments"),
        F.first("payment_type", ignorenulls=True).alias("primary_payment_type"),
        # One-Hot Encoding binaire (faible cardinalité = 4 moyens de paiement, évite l'ordre artificiel du Label Encoding)
        F.max(F.when(F.col("payment_type") == "credit_card", 1).otherwise(0)).alias("pay_credit_card"),
        F.max(F.when(F.col("payment_type") == "boleto", 1).otherwise(0)).alias("pay_boleto"),
        F.max(F.when(F.col("payment_type") == "voucher", 1).otherwise(0)).alias("pay_voucher"),
        F.max(F.when(F.col("payment_type") == "debit_card", 1).otherwise(0)).alias("pay_debit_card"),
    )
)

reviews_per_order = (
    reviews_bz
    .filter(F.col("review_score").between(1, 5))
    .withColumn(
        "has_comment",
        F.when(clean_str("review_comment_message").isNotNull() | clean_str("review_comment_title").isNotNull(), 1).otherwise(0),
    )
    .groupBy("order_id")
    .agg(
        F.round(F.avg("review_score"), 2).alias("review_score"),
        F.max("has_comment").alias("has_review_comment"),
    )
)

# 3. Assemblage de `df_silver` : Features temporelles ISO (§2.3.2), Flags MAR/MNAR & Confrontation IQR vs Z-Score (§2.4.2)
count_before_join = orders_typed.count()

df_silver_base = (
    orders_typed
    .filter((F.year("order_purchase_timestamp") > 1970) & (F.col("delivery_days").isNull() | (F.col("delivery_days") >= 0)))
    .withColumn("order_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .withColumn("is_weekend", F.when(F.dayofweek("order_purchase_timestamp").isin(1, 7), F.lit(1)).otherwise(F.lit(0)))
    .withColumn("is_delivered", F.when(F.col("order_delivered_customer_date").isNotNull(), F.lit(1)).otherwise(F.lit(0)))
    .withColumn(
        "delay_days",
        F.round((F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_estimated_delivery_date")) / 86400.0, 2),
    )
    .withColumn("is_late", F.when(F.col("delay_days") > 0, F.lit(1)).otherwise(F.lit(0)))
    .join(customers_clean, on="customer_id", how="left")
    .join(items_per_order, on="order_id", how="inner")
    .join(payments_per_order, on="order_id", how="left")
    .join(reviews_per_order, on="order_id", how="left")
    .fillna({
        "category": "non_communique",
        "primary_payment_type": "non_communique",
        "pay_credit_card": 0, "pay_boleto": 0, "pay_voucher": 0, "pay_debit_card": 0,
        "has_review_comment": 0,
    })
)

# Calcul des bornes IQR vs Z-Score (§2.4.2) sur `delivery_days` et `total_amount`
stats_row = df_silver_base.select(
    F.mean("delivery_days").alias("deliv_mean"),
    F.stddev("delivery_days").alias("deliv_std"),
    F.percentile_approx("delivery_days", 0.25).alias("deliv_q1"),
    F.percentile_approx("delivery_days", 0.50).alias("deliv_med"),
    F.percentile_approx("delivery_days", 0.75).alias("deliv_q3"),
    F.percentile_approx("total_amount", 0.25).alias("amt_q1"),
    F.percentile_approx("total_amount", 0.50).alias("amt_med"),
    F.percentile_approx("total_amount", 0.75).alias("amt_q3"),
).first()

deliv_iqr = float(stats_row["deliv_q3"] - stats_row["deliv_q1"])
deliv_upper_iqr = round(float(stats_row["deliv_q3"]) + 1.5 * deliv_iqr, 2)
deliv_upper_z = round(float(stats_row["deliv_mean"]) + 3.0 * float(stats_row["deliv_std"]), 2)
amt_med = float(stats_row["amt_med"])
amt_iqr = float(stats_row["amt_q3"] - stats_row["amt_q1"]) or 1.0

print(f"\n=== Confrontation Seuils d'Outliers (§2.4.2) sur `delivery_days` ===")
print(f" - Seuil IQR robuste (Q3 + 1.5*IQR) : {deliv_upper_iqr:.2f} jours (Médiane={stats_row['deliv_med']:.2f}j, IQR={deliv_iqr:.2f}j)")
print(f" - Seuil Z-Score classique (μ + 3σ) : {deliv_upper_z:.2f} jours (gonflé par les retards extrêmes !)")

df_silver = (
    df_silver_base
    .withColumn("is_delivery_outlier_iqr", F.when(F.col("delivery_days") > deliv_upper_iqr, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("total_amount_robust_scaled", F.round((F.col("total_amount") - F.lit(amt_med)) / F.lit(amt_iqr), 4))
)

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(SILVER_MAIN_PATH)

silver_count = spark.read.format("delta").load(SILVER_MAIN_PATH).count()
b2b_match_pct = spark.read.format("delta").load(SILVER_MAIN_PATH).select(F.round(100.0 * F.avg("has_b2b_profile"), 2)).first()[0]
assert silver_count <= count_before_join, (
    f"Alerte explosion N-M : {silver_count:,} lignes après jointure > {count_before_join:,} lignes avant jointure !"
)

print(f"\n[SILVER OK] {SILVER_MAIN_PATH} : {silver_count:,} commandes (<= {count_before_join:,}, 0 explosion N-M | match B2B = {b2b_match_pct}%)")
print("\n=== Aperçu des valeurs nettoyées et enrichies dans `/lakehouse/silver/orders` ===")
spark.read.format("delta").load(SILVER_MAIN_PATH).select(
    "order_id", "customer_state", "category", "category_is_missing",
    "seller_lead_origin", "order_status", "is_delivered", "is_weekend",
    "total_amount", "total_amount_robust_scaled", "pay_credit_card",
    "delivery_days", "is_late", "is_delivery_outlier_iqr", "review_score", "has_review_comment"
).show(10, truncate=False)

# Contrôle qualité post-wrangling : vérification des valeurs nulles résiduelles sur `/lakehouse/silver/orders`
print("=== Contrôle final post-Wrangling des valeurs NULL sur `/lakehouse/silver/orders` ===")
silver_check_df = spark.read.format("delta").load(SILVER_MAIN_PATH)
key_verify_cols = [
    "order_id", "customer_id", "customer_state", "category", "category_is_missing",
    "seller_lead_origin", "total_amount", "total_amount_robust_scaled", "is_delivered",
    "delivery_days", "has_review_comment"
]
silver_check_df.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in key_verify_cols]).show(truncate=False)

## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [ ]:
# 1. Lecture directe du journal transactionnel JSON sur HDFS
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)

# 2. TODO : Tester le Schema Enforcement avec un DataFrame au mauvais schéma dans un bloc try/except
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    print("Schema Enforcement validé — écriture invalide bloquée :", str(e).splitlines()[0])

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [ ]:
# Simulation de la confirmation de livraison de 5 commandes 'shipped' via MERGE INTO transactionnel
shipped_updates = (
    spark.read.format("delta").load(SILVER_MAIN_PATH)
    .filter(F.col("order_status") == "shipped")
    .limit(5)
    .select(
        "order_id",
        F.lit("delivered").alias("new_status"),
        F.lit(1).alias("new_is_delivered"),
        F.lit(12.5).alias("new_delivery_days"),
        F.lit(0).alias("new_is_late"),
    )
)

target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
(
    target_delta.alias("target")
    .merge(
        shipped_updates.alias("updates"),
        "target.order_id = updates.order_id",
    )
    .whenMatchedUpdate(set={
        "order_status": "updates.new_status",
        "is_delivered": "updates.new_is_delivered",
        "delivery_days": "updates.new_delivery_days",
        "is_late": "updates.new_is_late",
    })
    .execute()
)

history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")
history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)
assert history_df.count() >= 2, "Exécutez au moins une opération MERGE/UPDATE pour créer une version >= 1 !"

# Comparaison Time Travel : Version initiale (0) vs Version courante
v0_shipped = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_MAIN_PATH).filter(F.col("order_status") == "shipped").count()
now_shipped = spark.read.format("delta").load(SILVER_MAIN_PATH).filter(F.col("order_status") == "shipped").count()
print(f"Commandes 'shipped' en VERSION AS OF 0 : {v0_shipped:,}")
print(f"Commandes 'shipped' en VERSION COURANTE : {now_shipped:,} (écart = {v0_shipped - now_shipped})")

## 4. Analyse Exploratoire des Données (EDA) & Visualisation

> *« On ne peut jamais comprendre véritablement un jeu de données sans le regarder. Les statistiques descriptives seules mentent par omission (piège du Quartet d'Anscombe). »* — John Tukey ([Chapitre 4 — EDA](https://datascience.apti.space/cours/4_eda/))

Une fois la couche **Silver** fiabilisée (`/lakehouse/silver/orders`), nous menons l'**enquête exploratoire** et **visuelle** en suivant rigoureusement les deux chapitres du cours :
- **[Chapitre 3 — Visualisation des Données](https://datascience.apti.space/cours/3_visualisation/)** : grammaire des graphiques **1D** (*Histogramme continu* vs *Bar Chart discret* vs *Donut Chart $\le 3$ modalités*), arbre décisionnel **2D** (*Line Plot chronologique ordonné*, *Boxplot IQR*) et projections **multidimensionnelles** (*Bubble Plot 4D sans perspective 3D trompeuse*, *Heatmap*, réduction de dimension par **ACP / PCA**).
- **[Chapitre 4 — Analyse Exploratoire des Données (EDA)](https://datascience.apti.space/cours/4_eda/)** : confrontation des indicateurs sensibles vs robustes (**Moyenne vs Médiane**, **Écart-type vs IQR**), tris croisés (`crosstab`), segmentation distribuée (*Split-Apply-Combine* `.groupBy().agg()`), discrétisation métier vs fréquentielle (`pd.cut` vs `pd.qcut`), et confrontation des corrélations **Pearson ($r$) vs Spearman ($\rho$)** (*Corrélation $\neq$ Causalité*).

> **Règle d'or Big Data (PySpark $\rightarrow$ Driver)** : ne jamais appeler `.toPandas()` sur une table brute volumineuse ! Toujours **agréger** (`.groupBy().agg()`) ou **projeter + échantillonner** (`.select(...).sample(...)`) sur le cluster Spark avant le rendu `matplotlib` / `seaborn`.

### 4.1 Statistiques Descriptives Robustes, Tris Croisés & Distributions Unidimensionnelles (1D)

Conformément au **[Chapitre 4 (§4.2 & §4.8)](https://datascience.apti.space/cours/4_eda/)** et au **[Chapitre 3 (§3.1)](https://datascience.apti.space/cours/3_visualisation/)** :
1. **Tendance centrale & Dispersion** : confrontons la **Moyenne** et l'**Écart-type** (très sensibles aux paniers et délais extrêmes) à la **Médiane** (`PERCENTILE_APPROX(..., 0.5)`) et à l'**Écart Interquartile (`IQR = Q3 - Q1`)**, complétés par le **Mode** pour les variables qualitatives.
2. **Tri croisé (`pd.crosstab`)** : vérifions scientifiquement l'impact d'un retard de livraison (`is_late`) sur la probabilité d'obtenir un avis client très négatif ($\le 2\star$).
3. **Grammaire des distributions 1D** :
   - **Histogramme** (barres *adjacentes*) pour la variable quantitative continue `delivery_days`, illustrant l'asymétrie à droite (*Moyenne > Médiane*).
   - **Diagramme en barres** (barres *espacées*, triées par fréquence) pour le Top 10 des catégories discrètes.
   - **Donut Chart** strictement limité à **2 modalités** (*À l'heure* vs *En retard*), centre évidé avec KPI global, sans aucun effet 3D.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

warnings.filterwarnings("ignore", message=".*pandas >= 3.0.0.*")
sns.set_theme(style="whitegrid", context="notebook")
SILVER_MAIN_PATH = globals().get("SILVER_MAIN_PATH", "/lakehouse/silver/orders")

silver_eda = spark.read.format("delta").load(SILVER_MAIN_PATH)
delivered_eda = silver_eda.filter(F.col("is_delivered") == 1).cache()

# 1. Confrontation Moyenne / Écart-type vs Médiane / IQR sur PySpark (Chap. 4 §4.2)
num_metrics = ["total_amount", "freight_amount", "delivery_days", "delay_days", "review_score"]
summary_rows = []
for col_name in num_metrics:
    st = delivered_eda.select(
        F.round(F.mean(col_name), 2).alias("mean"),
        F.round(F.stddev(col_name), 2).alias("std"),
        F.round(F.percentile_approx(col_name, 0.25), 2).alias("q1"),
        F.round(F.percentile_approx(col_name, 0.50), 2).alias("median"),
        F.round(F.percentile_approx(col_name, 0.75), 2).alias("q3"),
    ).first()
    summary_rows.append({
        "variable": col_name,
        "moyenne": float(st["mean"]),
        "mediane_robuste": float(st["median"]),
        "ecart_moy_med": round(float(st["mean"]) - float(st["median"]), 2),
        "ecart_type": float(st["std"]),
        "IQR (Q3 - Q1)": round(float(st["q3"]) - float(st["q1"]), 2),
    })

print("=== 1. TENDANCE CENTRALE & DISPERSION : PARAMÉTRIQUE vs ROBUSTE (Commandes livrées) ===")
print(pd.DataFrame(summary_rows).to_string(index=False))

# Mode des variables catégorielles majeures
top_cat_mode = delivered_eda.groupBy("category").count().orderBy(F.desc("count")).first()
top_pay_mode = delivered_eda.groupBy("primary_payment_type").count().orderBy(F.desc("count")).first()
print(f"\nModes catégoriels -> Catégorie dominante : '{top_cat_mode['category']}' ({top_cat_mode['count']:,} cmd) "
      f"| Paiement dominant : '{top_pay_mode['primary_payment_type']}' ({top_pay_mode['count']:,} cmd)")

# 2. Tri croisé (Crosstab - Chap. 4 §4.8) : Retard logistique x Insatisfaction client (Note <= 2)
ct_agg = (
    delivered_eda
    .filter(F.col("review_score").isNotNull())
    .withColumn("statut_delai", F.when(F.col("is_late") == 1, "En retard").otherwise("À l'heure"))
    .withColumn("satisfaction", F.when(F.col("review_score") <= 2, "Insatisfait (1-2★)").otherwise("Satisfait (3-5★)"))
    .groupBy("statut_delai", "satisfaction")
    .count()
    .toPandas()
)
crosstab_pct = (
    pd.crosstab(
        index=ct_agg["statut_delai"],
        columns=ct_agg["satisfaction"],
        values=ct_agg["count"],
        aggfunc="sum",
        normalize="index",
    ) * 100
).round(2)
print("\n=== 2. TRI CROISÉ (% par ligne) : IMPACT DU RETARD SUR LA SATISFACTION CLIENT ===")
print(crosstab_pct.to_string())

# 3. Visualisations Unidimensionnelles 1D (Chap. 3 §3.1) : Histogramme continu, Bar Chart trié, Donut (2 modalités)
deliv_sample_pd = (
    delivered_eda
    .filter(F.col("delivery_days") <= 60)
    .select("delivery_days")
    .sample(fraction=0.15, seed=42)
    .toPandas()
)
deliv_mean = summary_rows[2]["moyenne"]
deliv_med = summary_rows[2]["mediane_robuste"]

top10_cat_pd = (
    delivered_eda
    .filter(F.col("category") != "non_communique")
    .groupBy("category")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
    .toPandas()
)

late_counts_pd = (
    delivered_eda
    .groupBy("is_late")
    .count()
    .orderBy("is_late")
    .toPandas()
)

fig, axes = plt.subplots(1, 3, figsize=(19, 5.2))

# (A) Histogramme continu : barres adjacentes + Moyenne vs Médiane
sns.histplot(deliv_sample_pd["delivery_days"], bins=35, kde=True, color="#2b8cbe", edgecolor="white", ax=axes[0])
axes[0].axvline(deliv_mean, color="#d7301f", linestyle="--", linewidth=2, label=f"Moyenne = {deliv_mean:.1f} j")
axes[0].axvline(deliv_med, color="#238b45", linestyle="-", linewidth=2.2, label=f"Médiane = {deliv_med:.1f} j")
axes[0].set_title("1D Continu — Histogramme de `delivery_days` (≤ 60 j)", fontweight="bold")
axes[0].set_xlabel("Délai de livraison réel (jours)")
axes[0].set_ylabel("Fréquence (échantillon 15 %)")
axes[0].legend()

# (B) Diagramme en barres discret : barres espacées triées par fréquence
sns.barplot(data=top10_cat_pd, x="count", y="category", hue="category", palette="Blues_r", legend=False, ax=axes[1])
axes[1].set_title("1D Discret — Top 10 des catégories commandées", fontweight="bold")
axes[1].set_xlabel("Nombre de commandes livrées")
axes[1].set_ylabel("")

# (C) Donut Chart (strictement 2 modalités : À l'heure vs En retard)
labels_donut = ["À l'heure", "En retard"]
vals_donut = late_counts_pd["count"].tolist()
ontime_pct = 100.0 * vals_donut[0] / sum(vals_donut)
axes[2].pie(
    vals_donut,
    labels=labels_donut,
    autopct="%1.1f%%",
    startangle=90,
    colors=["#2ca25f", "#de2d26"],
    wedgeprops={"width": 0.42, "edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 10.5, "fontweight": "bold"},
)
axes[2].text(0, 0, f"{ontime_pct:.1f}%\nà l'heure", ha="center", va="center", fontsize=12, fontweight="bold")
axes[2].set_title("1D Proportions — Respect du délai estimé (Donut ≤ 3 mod.)", fontweight="bold")

plt.tight_layout()
plt.show()

### 4.2 Segmentation (*Split-Apply-Combine*), Discrétisation (`cut` vs `qcut`) & Visualisations 2D

Appliquons le paradigme **Split-Apply-Combine** (**[Chapitre 4 §4.4](https://datascience.apti.space/cours/4_eda/)**) et l'**arbre décisionnel 2D** (**[Chapitre 3 §3.3](https://datascience.apti.space/cours/3_visualisation/)**) :
1. **Agrégation vectorisée `.groupBy().agg()`** : compilée nativement par Catalyst en Spark (bien plus performante qu'un `pivot_table` séquentiel), avec **tri chronologique strict** (`orderBy("order_month")`) sur les mois complets ($\ge 100$ commandes) avant de tracer un **Line Plot**.
2. **Discrétisation (*Binning*) : `pd.cut` vs `pd.qcut`** :
   - **`pd.cut` (intervalles métier fixes)** sur `delivery_days` (`0–7j Express`, `8–15j Standard`, `16–30j Lent`, `>30j Critique`) pour identifier le seuil de décrochage de la satisfaction client.
   - **`pd.qcut` (quantiles d'effectifs égaux `Q1..Q4`)** sur `total_amount` pour répartir exactement 25 % des commandes par segment tarifaire malgré la forte asymétrie des prix.
3. **Numérique + Catégoriel $\rightarrow$ Boxplot** : comparaison de `delivery_days` entre les **5 macro-régions brésiliennes** (*Sudeste*, *Sul*, *Centro-Oeste*, *Nordeste*, *Norte*), synthétisant médiane, boîte IQR et moustaches ($1{,}5 \times \text{IQR}$).

In [ ]:
# 1. Split-Apply-Combine distribué par mois (Chap. 4 §4.4.1) — tri chronologique strict !
monthly_pd = (
    delivered_eda
    .groupBy("order_month")
    .agg(
        F.count("*").alias("nb_orders"),
        F.round(100.0 * F.avg("is_late"), 2).alias("late_rate_pct"),
        F.round(F.avg("review_score"), 2).alias("avg_review"),
    )
    .filter(F.col("nb_orders") >= 100)
    .orderBy("order_month")
    .toPandas()
)

# Projection échantillonnée avec macro-région brésilienne pour le Binning et le Boxplot régional
region_expr = (
    F.when(F.col("customer_state").isin("SP", "RJ", "MG", "ES"), "Sudeste")
    .when(F.col("customer_state").isin("PR", "SC", "RS"), "Sul")
    .when(F.col("customer_state").isin("DF", "GO", "MT", "MS"), "Centro-Oeste")
    .when(F.col("customer_state").isin("BA", "PE", "CE", "RN", "PB", "AL", "SE", "MA", "PI"), "Nordeste")
    .otherwise("Norte")
)

sample_2d_pd = (
    delivered_eda
    .withColumn("region", region_expr)
    .select("region", "delivery_days", "total_amount", "review_score", "is_late")
    .dropna(subset=["delivery_days", "total_amount", "review_score"])
    .sample(fraction=0.15, seed=42)
    .toPandas()
)

# 2. Confrontation Discrétisation : `pd.cut` (seuils métier fixes) vs `pd.qcut` (quantiles d'effectifs égaux)
sample_2d_pd["tranche_delai_cut"] = pd.cut(
    sample_2d_pd["delivery_days"],
    bins=[0, 7, 15, 30, 300],
    labels=["Express (0-7j)", "Standard (8-15j)", "Lent (16-30j)", "Critique (>30j)"],
    include_lowest=True,
)
sample_2d_pd["quartile_montant_qcut"] = pd.qcut(
    sample_2d_pd["total_amount"],
    q=4,
    labels=["Q1 Éco", "Q2 Moyen-", "Q3 Moyen+", "Q4 Premium"],
)

print("=== 1. DISCRÉTISATION MÉTIER `pd.cut` SUR `delivery_days` (Effectifs inégaux, seuils lisibles) ===")
cut_summary = sample_2d_pd.groupby("tranche_delai_cut", observed=False).agg(
    nb_echantillon=("review_score", "count"),
    note_moyenne=("review_score", "mean"),
    taux_retard_pct=("is_late", lambda s: 100.0 * s.mean()),
).round(2)
print(cut_summary.to_string())

print("\n=== 2. DISCRÉTISATION FRÉQUENTIELLE `pd.qcut` SUR `total_amount` (Effectifs équilibrés à 25 %) ===")
qcut_summary = sample_2d_pd.groupby("quartile_montant_qcut", observed=False).agg(
    nb_echantillon=("total_amount", "count"),
    montant_min=("total_amount", "min"),
    montant_max=("total_amount", "max"),
    note_moyenne=("review_score", "mean"),
).round(2)
print(qcut_summary.to_string())

# 3. Visualisations Bidimensionnelles 2D (Chap. 3 §3.3) : Line Plot temporel + Boxplot régional
fig, axes = plt.subplots(1, 2, figsize=(18, 5.5))

# (A) Line Plot temporel strictement ordonné (Volume + Taux de retard sur double axe lisible)
ax_vol = axes[0]
ax_late = ax_vol.twinx()
ax_vol.plot(monthly_pd["order_month"], monthly_pd["nb_orders"], marker="o", color="#1f78b4", linewidth=2.2, label="Commandes livrées")
ax_late.plot(monthly_pd["order_month"], monthly_pd["late_rate_pct"], marker="s", linestyle="--", color="#e31a1c", linewidth=2, label="Taux de retard (%)")
ax_vol.set_title("2D Temporel — Croissance mensuelle & Tensions logistiques", fontweight="bold")
ax_vol.set_xlabel("Mois de commande (`order_month` trié)")
ax_vol.set_ylabel("Volume de commandes", color="#1f78b4")
ax_late.set_ylabel("Commandes en retard (%)", color="#e31a1c")
ax_vol.tick_params(axis="x", rotation=45)
ax_late.grid(False)

lines_1, labels_1 = ax_vol.get_legend_handles_labels()
lines_2, labels_2 = ax_late.get_legend_handles_labels()
ax_vol.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")

# (B) Boxplot comparatif par macro-région brésilienne (filtré <= 50j pour lisibilité de la boîte IQR)
order_regions = ["Sudeste", "Sul", "Centro-Oeste", "Nordeste", "Norte"]
sns.boxplot(
    data=sample_2d_pd[sample_2d_pd["delivery_days"] <= 50],
    x="region",
    y="delivery_days",
    hue="region",
    order=order_regions,
    palette="Set2",
    legend=False,
    ax=axes[1],
)
axes[1].set_title("2D Numérique × Catégoriel — Boxplot `delivery_days` par Région", fontweight="bold")
axes[1].set_xlabel("Macro-région du client (Brésil)")
axes[1].set_ylabel("Délai de livraison (jours, tronqué à 50 j)")

plt.tight_layout()
plt.show()

### 4.3 Analyse Multivariée : Pearson vs Spearman, Pairplot, Bubble Plot 4D & ACP (PCA)

Terminons par l'exploration des interactions complexes (**[Chapitre 4 §4.3](https://datascience.apti.space/cours/4_eda/)** & **[Chapitre 3 §3.4](https://datascience.apti.space/cours/3_visualisation/)**) :
1. **Heatmaps de Corrélation : Pearson ($r$) vs Spearman ($\rho$)** :
   - **Pearson ($r$)** capture uniquement la colinéarité *linéaire* et se laisse fausser par les paniers extrêmes.
   - **Spearman ($\rho$)** opère sur les **rangs** : il capte toute relation *monotone* et résiste aux outliers.
   - **Attention — Corrélation $\neq$ Causalité** : la corrélation positive entre `freight_amount` et `delivery_days` ne signifie pas que payer des frais de port élevés ralentit le transporteur ; tous deux découlent d'une **variable de confusion (*confounding factor*)** : l'**éloignement géographique** (clients du *Nordeste* / *Norte* éloignés du hub logistique de *São Paulo*).
2. **Pairplot (`sns.pairplot`) en échelle logarithmique (`np.log1p`)** : vue panoramique croisant nuages bivariés et densités `KDE` sur la diagonale, colorée par statut de ponctualité (`hue`), après transformation `log(1 + x)` des grandeurs asymétriques (`items_amount`, `freight_amount`, `delivery_days`) pour décompresser les valeurs écrasées près de zéro.
3. **Bubble Plot 4D vs Projection ACP / PCA (`StandardScaler` + `PCA(n_components=2)`)** :
   - Plutôt qu'un nuage 3D souffrant d'**occlusion** et de **distorsion de perspective** (§3.4.7), le **Bubble Plot** encode **4 dimensions** sur un plan 2D (`X`, `Y`, `Taille`, `Couleur`).
   - Pour synthétiser **6 dimensions simultanées** décrivant les Top 25 catégories produits, nous appliquons une **Standardisation obligatoire (`StandardScaler`)** suivie d'une **ACP (`PCA`)** projetant le maximum de variance sur le plan factoriel `(PC1, PC2)`.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. Confrontation des matrices de corrélation Pearson (r) vs Spearman (rho) (Chap. 4 §4.3.2 & Chap. 3 §3.4.5)
corr_cols = ["items_amount", "freight_amount", "delivery_days", "delay_days", "review_score"]
corr_sample_pd = (
    delivered_eda
    .select(corr_cols + ["is_late"])
    .dropna()
    .sample(fraction=0.06, seed=42)
    .toPandas()
)

pearson_corr = corr_sample_pd[corr_cols].corr(method="pearson")
spearman_corr = corr_sample_pd[corr_cols].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
sns.heatmap(pearson_corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5, ax=axes[0])
axes[0].set_title("Corrélation de Pearson (r — linéaire, sensible aux outliers)", fontweight="bold")

sns.heatmap(spearman_corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5, ax=axes[1])
axes[1].set_title("Corrélation de Spearman (ρ — sur rangs, monotone & robuste)", fontweight="bold")
plt.tight_layout()
plt.show()

# 2. Pairplot panoramique bivarié + KDE en échelle log1p (Chap. 4 §4.3.1 & Chap. 2 §2.4.1) sur n = 1 000
pairplot_pd = corr_sample_pd.sample(n=min(1000, len(corr_sample_pd)), random_state=42).copy()
pairplot_pd["log1p(items_amount)"] = np.log1p(pairplot_pd["items_amount"])
pairplot_pd["log1p(freight_amount)"] = np.log1p(pairplot_pd["freight_amount"])
pairplot_pd["log1p(delivery_days)"] = np.log1p(pairplot_pd["delivery_days"])
pairplot_pd["Ponctualité"] = np.where(pairplot_pd["is_late"] == 1, "En retard", "À l'heure")

g = sns.pairplot(
    pairplot_pd[["log1p(items_amount)", "log1p(freight_amount)", "log1p(delivery_days)", "review_score", "Ponctualité"]],
    hue="Ponctualité",
    palette={"À l'heure": "#2ca25f", "En retard": "#de2d26"},
    diag_kind="kde",
    plot_kws={"alpha": 0.45, "s": 22},
    height=2.1,
)
g.figure.suptitle("Pairplot Multivarié (échelle log1p) & Densités KDE selon la Ponctualité (n = 1 000)", y=1.02, fontweight="bold")
plt.show()

# 3. Profilage 6D des Top 25 catégories -> Bubble Plot 4D & Réduction de dimension par ACP / PCA (Chap. 3 §3.4.2 & §3.4.6)
cat_profiles_pd = (
    delivered_eda
    .filter(F.col("category") != "non_communique")
    .groupBy("category")
    .agg(
        F.count("*").alias("nb_orders"),
        F.round(F.avg("items_amount"), 2).alias("avg_price"),
        F.round(F.avg("freight_amount"), 2).alias("avg_freight"),
        F.round(F.percentile_approx("delivery_days", 0.50), 2).alias("median_delivery_days"),
        F.round(100.0 * F.avg("is_late"), 2).alias("late_rate_pct"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score"),
        F.round(100.0 * F.avg("has_b2b_profile"), 2).alias("b2b_share_pct"),
    )
    .orderBy(F.desc("nb_orders"))
    .limit(25)
    .toPandas()
)

pca_features = ["avg_price", "avg_freight", "median_delivery_days", "late_rate_pct", "avg_review_score", "b2b_share_pct"]
X_scaled = StandardScaler().fit_transform(cat_profiles_pd[pca_features])  # Standardisation obligatoire avant ACP !
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
cat_profiles_pd["PC1"] = coords[:, 0]
cat_profiles_pd["PC2"] = coords[:, 1]

var_pc1, var_pc2 = pca.explained_variance_ratio_ * 100
var_total = var_pc1 + var_pc2
print(f"=== BILAN ACP (6D -> 2D sur Top 25 catégories) : PC1 = {var_pc1:.1f}% | PC2 = {var_pc2:.1f}% | Variance conservée = {var_total:.1f}% ===")
loadings_df = pd.DataFrame(pca.components_.T, index=pca_features, columns=["Poids_PC1", "Poids_PC2"]).round(3)
print(loadings_df.to_string())

fig, axes = plt.subplots(1, 2, figsize=(18, 6.2))

# (A) Bubble Plot 4D sur plan 2D (X = délai médian, Y = note moyenne, Taille = volume, Couleur = prix moyen)
sc_bubble = axes[0].scatter(
    cat_profiles_pd["median_delivery_days"],
    cat_profiles_pd["avg_review_score"],
    s=cat_profiles_pd["nb_orders"] / 18.0,
    c=cat_profiles_pd["avg_price"],
    cmap="viridis",
    alpha=0.8,
    edgecolors="black",
    linewidth=1,
)
for _, row in cat_profiles_pd.head(10).iterrows():
    axes[0].annotate(row["category"], (row["median_delivery_days"], row["avg_review_score"]),
                     xytext=(5, 4), textcoords="offset points", fontsize=8.5)
plt.colorbar(sc_bubble, ax=axes[0], label="Panier moyen hors port (BRL)")
axes[0].set_title("Bubble Plot 4D — Délai × Satisfaction × Volume (taille) × Prix (couleur)", fontweight="bold")
axes[0].set_xlabel("Délai médian de livraison (jours)")
axes[0].set_ylabel("Note client moyenne (★)")

# (B) Plan factoriel ACP 2D (PC1 vs PC2)
sc_pca = axes[1].scatter(
    cat_profiles_pd["PC1"],
    cat_profiles_pd["PC2"],
    s=cat_profiles_pd["nb_orders"] / 20.0,
    c=cat_profiles_pd["avg_review_score"],
    cmap="RdYlGn",
    alpha=0.85,
    edgecolors="black",
    linewidth=1,
)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=0.9)
axes[1].axvline(0, color="gray", linestyle="--", linewidth=0.9)
for _, row in cat_profiles_pd.iterrows():
    axes[1].annotate(row["category"], (row["PC1"], row["PC2"]),
                     xytext=(4, 4), textcoords="offset points", fontsize=8)
plt.colorbar(sc_pca, ax=axes[1], label="Note client moyenne (★)")
axes[1].set_title(f"Projection ACP (6D → 2D) — {var_total:.1f}% de variance capturée", fontweight="bold")
axes[1].set_xlabel(f"Axe Principal 1 — PC1 ({var_pc1:.1f}% de variance)")
axes[1].set_ylabel(f"Axe Principal 2 — PC2 ({var_pc2:.1f}% de variance)")

plt.tight_layout()
plt.show()

_ = delivered_eda.unpersist()